# Train a Window/Door Counter (YOLOv8)

Run this notebook on Google Colab with a GPU runtime (Runtime -> Change runtime type -> GPU).

This is a second, independent model from the segmentation one in `train_segmentation.ipynb`. Segmentation answers an *area* question ("what fraction of the facade is window?" -> WWR); this model answers a *count* question ("how many distinct windows/doors are there?"), which pixel-level segmentation can't do -- it doesn't separate individual object instances.

Unlike the segmentation pipeline, there's no hand-written training loop here: `ultralytics.YOLO` owns batching, augmentation, and checkpointing internally, so this notebook is much shorter.

## 1. Setup

In [ ]:
!pip install -q ultralytics

!git clone https://github.com/VickeyAryan/building-envelope-vision.git
%cd building-envelope-vision

import sys
sys.path.insert(0, 'src')

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 2. Dataset

324 images (276 train / 48 valid), 4 classes: building, door, gate, window. Vendored directly in the repo under `data/window_door_detection/` (Git LFS) -- see `ATTRIBUTION.md` there for the source (Roboflow Universe, "Facade Features", CC BY 4.0).

Known caveat: some multi-pane windows/glass doors are annotated as several sub-boxes rather than one box per window, so raw counts can run high on glass-heavy facades.

In [ ]:
from detect_model import CLASS_NAMES, resolve_data_yaml

DATA_YAML = 'data/window_door_detection/data.yaml'
print('classes:', CLASS_NAMES)

## 3. Train

In [ ]:
from pathlib import Path
import shutil
from detect_model import build_detector

EPOCHS = 100
IMGSZ = 416
BATCH = 16
OUTPUT_PATH = Path('models/best_detector.pt')
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

model = build_detector(pretrained=True)
model.train(
    data=resolve_data_yaml(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    name='facade_detector',
    exist_ok=True,
)

best_weights = model.trainer.save_dir / 'weights' / 'best.pt'
shutil.copy(best_weights, OUTPUT_PATH)
print(f'Best checkpoint copied to {OUTPUT_PATH}')

## 4. Evaluate on the validation split

In [ ]:
from detect_model import load_detector

best_model = load_detector(str(OUTPUT_PATH))
results = best_model.val(data=resolve_data_yaml(DATA_YAML), imgsz=IMGSZ, split='val')

print(f"mAP50: {results.box.map50:.4f}")
print(f"mAP50-95: {results.box.map:.4f}")
print('Per-class mAP50:')
for idx, ap50 in enumerate(results.box.ap50):
    name = CLASS_NAMES[idx] if idx < len(CLASS_NAMES) else str(idx)
    print(f'  {name:12s}: {ap50:.4f}')

## 5. Qualitative check: counts on a sample image

In [ ]:
import glob
import matplotlib.pyplot as plt
from PIL import Image
from detect_model import count_detections

sample_path = sorted(glob.glob('data/window_door_detection/valid/images/*'))[0]
sample_img = Image.open(sample_path).convert('RGB')

result = best_model.predict(sample_img, conf=0.25, verbose=False)[0]
counts = count_detections(result)
print('counts:', counts)

annotated = result.plot()  # ultralytics' own annotated-array helper (BGR)
plt.figure(figsize=(8, 8))
plt.imshow(annotated[..., ::-1])  # BGR -> RGB
plt.axis('off')
plt.show()

## 6. Download the checkpoint

Copy `models/best_detector.pt` back to your local `models/` directory so `app/streamlit_app.py` can use it.

In [ ]:
try:
    from google.colab import files
    files.download(str(OUTPUT_PATH))
except ImportError:
    print(f'Not running in Colab -- checkpoint already saved locally at {OUTPUT_PATH}')